# Batch processing template for CZI/LIF / CZI/LIF 批次分析模板

這份 notebook 示範如何把 napari-assistant 建立的單張影像流程，轉成 `.czi` 與 `.lif` 資料的批次分析。

**Teaching idea / 教學重點**
- 先在 napari 中完成單張影像流程
- 再把處理步驟貼到下方的迴圈中
- 最後輸出合併後的 CSV 結果表

> 這個版本特別把 **CZI 與 LIF 的批次差異** 拆開來寫：  
> - **CZI** 常見做法：`loop files`（逐檔處理）  
> - **LIF** 常見做法：`loop scenes`（同一檔中的多個 scene 逐一處理）

最大差別在於 **資料結構與讀檔方式**；  
後面的分析流程大多可以維持不變。


## 1. Import packages / 匯入套件

這些套件用來讀圖、分割、以及輸出量測結果。

**Main packages / 主要套件**
- `Path`: handle file paths / 處理路徑
- `AICSImage`: read `.czi` and `.lif` / 讀取 `.czi` 與 `.lif`
- `pyclesperanto`: GPU-style image processing / 影像處理
- `napari_simpleitk_image_processing`: label processing / label 處理
- `regionprops_table`: measurement / 量測
- `pandas`: save results as table / 表格整理

If `aicsimageio` is missing, the notebook can try to install it with `pip`.  
如果系統中沒有 `aicsimageio`，此 notebook 可以嘗試用 `pip` 自動安裝。
# 指令 pip install aicsimageio aicspylibczi>=3.1.1 fsspec>=2022.8.0

In [ ]:
from pathlib import Path
import pyclesperanto as cle
import napari_simpleitk_image_processing as nsitk
import pandas as pd
import numpy as np
from skimage.measure import regionprops_table
import napari

# Try importing aicsimageio / 嘗試匯入 aicsimageio
try:
    from aicsimageio import AICSImage
    AICS_AVAILABLE = True
    print("aicsimageio is available ✅")
except ImportError:
    AICS_AVAILABLE = False
    print("⚠️ aicsimageio is not installed. CZI/LIF reading is unavailable.")
    print("⚠️ 尚未安裝 aicsimageio，目前無法讀取 CZI/LIF。")





# viewer = napari.Viewer()  # optional / 可選


## 2. Set input folder and file format / 設定資料夾與檔案格式

請把要分析的 `.czi` 或 `.lif` 影像放在 `data/batch` 資料夾中。

### Recommended usage / 建議用法
- **CZI**：資料夾中放多個 `.czi` 檔案，程式會逐檔處理  
- **LIF**：通常是一個 `.lif` 檔裡面包含多個 scene，程式會先開檔，再逐一處理 scene

這個範例模板預設會取出：
- `T=0`
- `C=0`
- 2D `YX` 影像

如果你的資料使用其他 channel、time point，或需要 Z projection，之後再修改即可。


In [ ]:
# Input folder / 輸入資料夾
input_dir = Path("data/batch2")
file_suffix = ".lif"   # 改成 ".lif" 可切換示範格式
results_list = []

# Basic reading parameters / 基本讀檔參數
time_index = 0
channel_index = 0

# Check folder / 檢查資料夾
if not input_dir.exists():
    print(f"找不到資料夾：{input_dir.absolute()}，請先建立它並放入圖片！")

def get_scene_names(path):
    """
    Return all scene names in one file.
    取得單一檔案中的所有 scene 名稱。
    """
    if not AICS_AVAILABLE:
        return []

    img = AICSImage(str(path))
    return list(img.scenes)

def load_image(path, scene_name=None, time_index=0, channel_index=0):
    """
    Read one CZI/LIF file and return:
    讀取單個 CZI/LIF 檔案，並回傳：

    1. 2D NumPy array
    2. pixel size Y (um/pixel)
    3. pixel size X (um/pixel)

    scene_name:
        - None: keep the current/default scene
        - str: switch to the specified scene first
    """
    if not AICS_AVAILABLE:
        print(f"⚠️ Skipping {path.name}: aicsimageio is not available / 尚未安裝 aicsimageio")
        return None, np.nan, np.nan

    img = AICSImage(str(path))

    if scene_name is not None:
        img.set_scene(scene_name)

    data = img.get_image_data("YX", T=time_index, C=channel_index)
    data = np.asarray(data).astype(np.float32)

    # Read physical pixel sizes from metadata / 從 metadata 讀取實際像素大小
    px = img.physical_pixel_sizes
    pixel_size_y = float(px.Y) if px.Y is not None else np.nan
    pixel_size_x = float(px.X) if px.X is not None else np.nan

    return data, pixel_size_y, pixel_size_x

from pathlib import Path

files = list(input_dir.glob(f"*{file_suffix}"))
print(files)
print(f"找到 {len(files)} 個檔案")


## 3. Main batch loop / 主要批次迴圈

這裡把 **CZI** 和 **LIF** 分成兩種常見模式：

### Mode A: CZI = loop files / CZI 常見做法：逐檔處理
1. Read one file / 讀取一個檔案  
2. Run the same workflow / 執行相同的處理流程  
3. Measure objects / 量測物件  
4. Append the result / 加到結果表中  

### Mode B: LIF = loop scenes / LIF 常見做法：逐 scene 處理
1. Open one `.lif` file / 開啟一個 `.lif` 檔  
2. List all scenes / 取得全部 scenes  
3. Run the same workflow for each scene / 對每個 scene 套用相同流程  
4. Append the result / 加到結果表中  

### Important notes / 重要提醒
- Keep the indentation inside the `for` loop.  
  貼上 assistant 程式碼時，要保留 `for` 迴圈內的縮排。
- Variable names may differ between students.  
  每個人從 assistant 產生的變數名稱可能不同。
- To avoid errors, always assign the final label image to `final_label_image`.  
  為了避免錯誤，請把最後的 label 結果指定給 `final_label_image`。
- The key idea of batch processing is not only `loop files`; sometimes it is `loop scenes`.  
  批次分析不一定只是逐檔，也可能是逐 scene。


In [ ]:
# =============================
# Workflow function
# =============================
def run_workflow_and_measure(image_data, source_name, pixel_size_y=np.nan, pixel_size_x=np.nan):

    # ---- Image processing ----
    image_gb = cle.gaussian_blur(image_data, None, 2.0, 2.0, 0)
    image_to = cle.threshold_otsu(image_gb)
    label_image = nsitk.touching_objects_labeling(image_to)
    final_label_image = cle.exclude_labels_on_edges(label_image, None, True, True, True)

    # ---- Measurement ----
    label_np = np.asarray(final_label_image).astype(np.int32)
    intensity_np = np.asarray(image_data).astype(np.float32)

    props = regionprops_table(
        label_np,
        intensity_image=intensity_np,
        properties=["label", "area", "mean_intensity", "centroid"]
    )

    df_single_image = pd.DataFrame(props)

    # ---- Add source ----
    df_single_image["source"] = source_name

    # ---- Metadata ----
    df_single_image["pixel_size_y_um"] = pixel_size_y
    df_single_image["pixel_size_x_um"] = pixel_size_x

    # ---- Convert to real units ----
    if not np.isnan(pixel_size_y) and not np.isnan(pixel_size_x):
        df_single_image["area_um2"] = df_single_image["area"] * pixel_size_y * pixel_size_x
        df_single_image["centroid_y_um"] = df_single_image["centroid-0"] * pixel_size_y
        df_single_image["centroid_x_um"] = df_single_image["centroid-1"] * pixel_size_x

    # ---- FINAL: reorder columns (最重要放最後) ----
    cols = ["source"] + [c for c in df_single_image.columns if c != "source"]
    df_single_image = df_single_image[cols]

    return df_single_image


# =============================
# Batch processing
# =============================
results_list = []

# ---- CZI: loop files ----
if file_suffix.lower() == ".czi":
    for img_path in input_dir.glob("*.czi"):

        image_data, pixel_size_y, pixel_size_x = load_image(
            img_path,
            scene_name=None,
            time_index=time_index,
            channel_index=channel_index,
        )

        if image_data is None:
            continue

        df_single_image = run_workflow_and_measure(
            image_data,
            img_path.name,
            pixel_size_y=pixel_size_y,
            pixel_size_x=pixel_size_x,
        )

        results_list.append(df_single_image)
        print(f"完成: {img_path.name} -> 細胞數: {len(df_single_image)}")


# ---- LIF: loop scenes ----
elif file_suffix.lower() == ".lif":

    for img_path in input_dir.glob("*.lif"):

        scene_names = get_scene_names(img_path)

        if len(scene_names) == 0:
            print(f"⚠️ 無法取得 scene：{img_path.name}")
            continue

        print(f"\n處理 {img_path.name}，共 {len(scene_names)} 個 scenes")

        for scene_name in scene_names:

            image_data, pixel_size_y, pixel_size_x = load_image(
                img_path,
                scene_name=scene_name,
                time_index=time_index,
                channel_index=channel_index,
            )

            if image_data is None:
                continue

            source_name = f"{img_path.name} | scene: {scene_name}"

            df_single_image = run_workflow_and_measure(
                image_data,
                source_name,
                pixel_size_y=pixel_size_y,
                pixel_size_x=pixel_size_x,
            )

            results_list.append(df_single_image)
            print(f"完成: {source_name} -> 細胞數: {len(df_single_image)}")

else:
    print(f"⚠️ 不支援的副檔名：{file_suffix}")


# =============================
# Merge & export
# =============================
if results_list:

    final_report = pd.concat(results_list, ignore_index=True)

    # 移除背景 label
    final_report = final_report[final_report["label"] > 0]

    # ---- 再保險排序一次（一定成功） ----
    cols = ["source"] + [c for c in final_report.columns if c != "source"]
    final_report = final_report[cols]

    final_report.to_csv("final_analysis.csv", index=False, encoding="utf_8_sig")

    print("\n🎉 任務完成！表格已存入 final_analysis.csv")
    display(final_report.head())

else:
    print("⚠️ No results were generated.")

## 4. Physical units from metadata / 從 metadata 轉成真實單位

`regionprops_table` 預設輸出的是 **pixel-based measurements**：
- `area` = number of pixels / pixel 數量
- `centroid` = pixel coordinates / pixel 座標

如果希望得到 **um / um²**，就要從影像 metadata 讀取 `pixel size`，再做單位換算。

### In this notebook / 在這份 notebook 中
- `load_image()` 會讀出 `pixel_size_y_um` 與 `pixel_size_x_um`
- `run_workflow_and_measure()` 會自動新增：
  - `area_um2`
  - `centroid_y_um`
  - `centroid_x_um`

### Formula / 換算公式
- `area_um2 = area * pixel_size_y * pixel_size_x`
- `centroid_y_um = centroid-0 * pixel_size_y`
- `centroid_x_um = centroid-1 * pixel_size_x`

> napari 可以正確顯示影像比例（scale），  
> 但真正的量測單位轉換，仍需要在 Python 中明確處理。


## 5. Notes for multi-dimensional files / 多維檔案注意事項

### CZI vs LIF / CZI 與 LIF 的差異
- **CZI**：常見情境是一個檔案一個 sample，因此常用 `loop files`
- **LIF**：常見情境是一個檔案裡有多個 scene，因此常用 `loop scenes`

### Scene / 場景
Some files contain more than one scene. In that case, list `img.scenes` first.  
有些檔案包含多個 scene，可先用 `img.scenes` 查看。

### Channel / 通道
If your nuclei signal is not in channel 0, change `channel_index`.  
如果你的 nuclei 不在第 0 個 channel，請修改 `channel_index`。

### Time / 時間點
If you want another time point, change `time_index`.  
如果要分析其他時間點，請修改 `time_index`。

### Z dimension / Z 軸
This template extracts a 2D image with `YX`. If your workflow needs one Z slice or a projection, add that step in the loader.  
這個模板用 `YX` 取出 2D 影像；如果你的流程需要單一 Z slice 或 projection，請在讀檔函式中加入該步驟。


## 6. Output and common problems / 輸出與常見問題

當迴圈結束後，所有量測結果會合併成一份 `final_analysis.csv`。

### Common problems / 常見問題
- **`aicsimageio` not installed / 未安裝 `aicsimageio`**  
  Install it manually with `pip`, for example:  
  `pip install aicsimageio aicspylibczi>=3.1.1 fsspec>=2022.8.0`
- **Indentation error / 縮排錯誤**  
  assistant 貼上的程式碼若在函式或迴圈外，容易出錯。
- **Wrong final variable / 最後變數名稱錯誤**  
  請確認 `final_label_image = ...` 指到最後的 label 結果。
- **Type mismatch in `regionprops_table` / `regionprops_table` 型別不一致**  
  在量測前，label image 與 intensity image 都先用 `np.asarray()` 轉成 NumPy array。
- **Wrong channel / 通道選錯**  
  如果影像看起來是空的或不對，請檢查 `channel_index`。
- **LIF scenes / LIF 多 scene 問題**  
  如果 `.lif` 裡有很多 scene，不要只處理第一個；可以逐一 loop scene。
